# 02 XGBoost

Run on a Colab T4 GPU runtime.

## Setup

In [ ]:
# Setup: in Colab, clone the repo and install packages. Locally, use the checkout.
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    if not os.path.exists('/content/credit-card-fraud-detection'):
        !git clone -q https://{token}@github.com/VanshSharmaPES/credit-card-fraud-detection.git /content/credit-card-fraud-detection
    %cd /content/credit-card-fraud-detection
    !pip install -q -r requirements/base.txt -r requirements/xgboost.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
!{sys.executable} src/data.py
sys.path.append('src')

In [ ]:
import json
import random
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from data import load_splits
from evaluate import evaluate_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Use the GPU when there is one. All timed runs should be on a Colab T4.
def gpu_name():
    if shutil.which('nvidia-smi') is None:
        return None
    out = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                         capture_output=True, text=True)
    return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else None

GPU = gpu_name()
DEVICE = 'cuda' if GPU else 'cpu'
HARDWARE = f'Colab {GPU}' if (GPU and IN_COLAB) else (GPU or 'CPU')
print('xgboost', xgb.__version__, '| device:', DEVICE, '| hardware:', HARDWARE)
if GPU is None:
    print('WARNING: no GPU found. Use Runtime > Change runtime type > T4 GPU for the timed run.')

FIG_DIR, MODEL_DIR = Path('figures'), Path('models')
FIG_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': False,
})

## Load the data

In [ ]:
(X_train, y_train, X_val, y_val, X_test, y_test,
 rowid_val, rowid_test, amount_val, amount_test) = load_splits()

CLASS_WEIGHT = (y_train == 0).sum() / (y_train == 1).sum()
print('Train / val / test rows:', len(X_train), len(X_val), len(X_test))
print('Features:', X_train.shape[1])
print(f'Class weight (train normal / train fraud): {CLASS_WEIGHT:.4f}')
assert X_train.shape[1] == 30
assert round(CLASS_WEIGHT, 4) == 598.4190, "Class weight differs from the data.py summary"

## Train

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=2000, learning_rate=0.05, max_depth=6, scale_pos_weight=CLASS_WEIGHT,
    eval_metric='aucpr', early_stopping_rounds=50, device=DEVICE, random_state=SEED,
)
start = time.perf_counter()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
train_seconds = time.perf_counter() - start
print(f'Best iteration: {model.best_iteration} | val PR-AUC: {model.best_score:.4f} | {train_seconds:.1f} s')

## Predict

In [ ]:
# predict_proba uses the best iteration from early stopping
p_val = model.predict_proba(X_val)[:, 1]
start = time.perf_counter()
p_test = model.predict_proba(X_test)[:, 1]
predict_seconds = time.perf_counter() - start
print(f'Test prediction: {predict_seconds:.3f} s for {len(X_test):,} rows')

## Evaluate

In [ ]:
# Argument names follow the interface agreed for src/evaluate.py (plan Step 12).
evaluate_model(
    'xgboost',
    y_val=y_val, p_val=p_val, y_test=y_test, p_test=p_test,
    rowid_val=rowid_val, rowid_test=rowid_test,
    train_seconds=train_seconds, predict_seconds=predict_seconds,
    hardware=HARDWARE,
    notes=f'best_iteration={model.best_iteration}, scale_pos_weight={CLASS_WEIGHT:.4f}',
)
results = json.loads(Path('results/xgboost.json').read_text())
print(json.dumps(results, indent=2))

In [ ]:
# Warning signs from plan Step 24
assert 0.5 < results['pr_auc'] < 0.98, f"PR-AUC {results['pr_auc']:.4f} is outside 0.5-0.98: check for a bug or leakage"
assert 0 < results['recall'] < 1, f"Recall is exactly {results['recall']}: check the threshold"
print('Sanity check passed: PR-AUC', round(results['pr_auc'], 4), '| recall', round(results['recall'], 4))

## Built-in feature importance (gain)

In [ ]:
gain = pd.Series(model.get_booster().get_score(importance_type='gain'))
gain = gain.reindex(X_train.columns, fill_value=0).sort_values()

fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(gain.index, gain.values, color='#2a78d6', height=0.7)
ax.set_xlabel('Average gain per split')
ax.set_title('XGBoost built-in feature importance (gain)')
ax.grid(axis='y', visible=False)
ax.margins(y=0.01)
fig.savefig(FIG_DIR / 'xgb_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
gain.sort_values(ascending=False).head(10)

## Save the model

In [ ]:
model.save_model(MODEL_DIR / 'xgboost.json')
print('Saved', MODEL_DIR / 'xgboost.json')

## SHAP

In [ ]:
import shap

# Explain only the trees that predict_proba uses (up to the best iteration)
best_booster = model.get_booster()[: model.best_iteration + 1]
X_shap = X_val.sample(2000, random_state=SEED)
print('SHAP sample:', len(X_shap), 'validation rows,', int(y_val.loc[X_shap.index].sum()), 'of them fraud')

def explain_with_xgboost(X):
    """Fallback from plan Step 26: XGBoost's own SHAP values (last column is the base value)."""
    contribs = best_booster.predict(xgb.DMatrix(X), pred_contribs=True)
    return shap.Explanation(values=contribs[:, :-1], base_values=contribs[:, -1],
                            data=X.values, feature_names=list(X.columns))

try:
    explain = shap.TreeExplainer(best_booster)
    shap_values = explain(X_shap)
    SHAP_SOURCE = f'shap {shap.__version__} TreeExplainer'
except Exception as err:
    # If this happens, log it in docs/decisions.md
    print('shap TreeExplainer failed, using pred_contribs instead:', repr(err))
    explain = explain_with_xgboost
    shap_values = explain(X_shap)
    SHAP_SOURCE = 'xgboost pred_contribs'

# SHAP values plus the base value must add up to the model's log-odds output
margin = model.predict(X_shap, output_margin=True)
gap = np.abs(shap_values.values.sum(axis=1) + shap_values.base_values - margin).max()
print('Source:', SHAP_SOURCE, '| largest additivity gap:', f'{gap:.2e}')
assert gap < 1e-2

### Summary on 2,000 validation rows

In [ ]:
shap.summary_plot(shap_values, X_shap, max_display=15, show=False)
fig = plt.gcf()
fig.suptitle('SHAP summary: XGBoost, 2,000 validation rows', fontweight='bold', y=1.01)
fig.savefig(FIG_DIR / 'xgb_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

mean_abs = pd.Series(np.abs(shap_values.values).mean(axis=0), index=X_shap.columns)
mean_abs.sort_values(ascending=False).head(10).rename('mean |SHAP|')

### One caught fraud explained

In [ ]:
# One fraud that the model caught at the chosen threshold.
# The lowest row_id is taken, so the case is not hand-picked.
threshold = results['threshold']
caught = (y_val.values == 1) & (p_val >= threshold)
print(f'Threshold {threshold:.4f}: {caught.sum()} of {int(y_val.sum())} validation frauds caught')
pos = np.flatnonzero(caught)[np.argmin(rowid_val.values[caught])]
case = X_val.iloc[[pos]]
print(f'Case row_id {rowid_val.iloc[pos]} | fraud probability {p_val[pos]:.4f}')

case_values = explain(case)
shap.plots.waterfall(case_values[0], max_display=12, show=False)
fig = plt.gcf()
fig.suptitle(f'Why row {rowid_val.iloc[pos]} was flagged (log-odds, probability {p_val[pos]:.4f})',
             fontweight='bold', y=1.02)
fig.savefig(FIG_DIR / 'xgb_shap_case.png', dpi=150, bbox_inches='tight')
plt.show()

## Permutation importance

In [ ]:
from evaluate import permutation_importance

perm = permutation_importance(lambda X: model.predict_proba(X)[:, 1], X_val, y_val, n_repeats=3)
perm.to_csv('results/perm_xgboost.csv', index=not isinstance(perm.index, pd.RangeIndex))
print('Saved results/perm_xgboost.csv')
perm.head(10)

## Package versions

In [ ]:
# Exact versions, for pinning requirements/base.txt and requirements/xgboost.txt
from importlib.metadata import version
for pkg in ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'imbalanced-learn', 'shap', 'xgboost']:
    print(f'{pkg}=={version(pkg)}')